# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lokeshtiwari723/Proto-ex/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes


The playbook turns the validated ranking output into a review queue for human decision-making.

Each ranked item should have:
- a rank showing review order
- an action describing what a reviewer should consider
- a reason code explaining why the item was ranked
- the model score used for ordering

The intended action is review and prioritization, not automatic content changes. A reviewer should inspect the page and supporting context before taking action.

The main reason codes are:
- high_model_priority: ranked highly by the Logistic Regression model
- baseline_visible: meets the Week-4 visibility threshold used by the action-score baseline
- baseline_stale: meets the Week-4 content-age threshold
- general_review: included for review but does not meet the specific baseline conditions

The queue is decision-support evidence. It does not establish that a page is declining, that refreshing a page will cause improvement, or that an action is guaranteed to work.

In [10]:
# Check the variables created by the ML-09 notebook
names = [
    name for name in globals()
    if any(word in name.lower() for word in [
        "model", "test", "train", "rank", "score", "after"
    ])
]

print("Relevant variables found:")
for name in sorted(names):
    print("-", name)

Relevant variables found:
- model
- model_df
- score_median
- test_df
- test_idx
- test_ranked
- train_df
- train_idx


### Ranked actions + reason codes

The queue ranks pages by model score and combines the model priority with the Week-4 baseline signals. Each row has an action and reason code so a reviewer can understand why the page appears in the queue.

High-priority pages are marked for review for refresh when the model and baseline signals support that action. Other pages can be marked as a review opportunity or general review.

The ranking is for human review prioritization, not an automatic decision.

In [11]:
# ML-10: Build the validated ranking output and action queue

import numpy as np
import pandas as pd
import duckdb

from google.colab import userdata
from huggingface_hub import HfApi

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# --------------------------------------------------
# 1. Load Hugging Face token
# --------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

api = HfApi(token=HF_TOKEN)
print("HF token loaded.")

# --------------------------------------------------
# 2. Connect to FlyRank warehouse
# --------------------------------------------------

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"

print("DuckDB connection ready.")

# --------------------------------------------------
# 3. February feature frame
# --------------------------------------------------

features = con.sql(f"""
WITH feb_page AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        SUM(gsc_sum_position)
            / NULLIF(SUM(gsc_impressions), 0) AS gsc_avg_position
    FROM {FEB}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions,
    f.gsc_clicks,
    f.gsc_avg_position,
    d.word_count,
    d.content_created_date
FROM feb_page AS f
LEFT JOIN {DIM} AS d
    ON f.client_hash_id = d.client_hash_id
    AND f.content_hash_id = d.content_hash_id
""").df()

# --------------------------------------------------
# 4. March label
# --------------------------------------------------

label_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS march_clicks,
    SUM(gsc_impressions) AS march_impressions
FROM {MAR}
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) >= 100
""").df()

label_df["march_ctr"] = (
    label_df["march_clicks"]
    / label_df["march_impressions"]
)

ctr_cutoff = label_df["march_ctr"].median()

label_df["label"] = (
    label_df["march_ctr"] > ctr_cutoff
).astype(int)

# --------------------------------------------------
# 5. Join February features to March label
# --------------------------------------------------

model_df = features.merge(
    label_df[
        [
            "client_hash_id",
            "content_hash_id",
            "march_ctr",
            "label"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_df["content_created_date"] = pd.to_datetime(
    model_df["content_created_date"],
    errors="coerce"
)

decision_date = pd.Timestamp("2026-02-28")

model_df["content_age_days"] = (
    decision_date
    - model_df["content_created_date"]
).dt.days

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "word_count",
    "content_age_days"
]

model_df = model_df.dropna(
    subset=feature_cols + ["label"]
).copy()

print("Rows available for modeling:", len(model_df))

# --------------------------------------------------
# 6. Honest client-grouped split
# --------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        model_df["label"],
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

client_overlap = len(
    set(train_df["client_hash_id"])
    & set(test_df["client_hash_id"])
)

print("Client overlap:", client_overlap)

# --------------------------------------------------
# 7. Train Logistic Regression
# --------------------------------------------------

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(
    train_df[feature_cols],
    train_df["label"]
)

# --------------------------------------------------
# 8. Create ranking scores
# --------------------------------------------------

test_ranked = test_df.copy()

test_ranked["model_score"] = model.predict_proba(
    test_ranked[feature_cols]
)[:, 1]

test_ranked = test_ranked.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

test_ranked["rank"] = (
    np.arange(len(test_ranked)) + 1
)

# --------------------------------------------------
# 9. Create human-readable action queue
# --------------------------------------------------

action_queue = test_ranked.copy()

action_queue["visible"] = (
    action_queue["gsc_impressions"] >= 100
)

action_queue["stale"] = (
    action_queue["content_age_days"] >= 180
)

score_median = action_queue["model_score"].median()

def make_reason(row):
    reasons = []

    if row["model_score"] >= score_median:
        reasons.append("high_model_priority")

    if row["visible"]:
        reasons.append("baseline_visible")

    if row["stale"]:
        reasons.append("baseline_stale")

    if not reasons:
        reasons.append("general_review")

    return ";".join(reasons)

action_queue["reason_code"] = action_queue.apply(
    make_reason,
    axis=1
)

action_queue["action"] = np.where(
    action_queue["stale"] & action_queue["visible"],
    "Review for refresh",
    np.where(
        action_queue["visible"],
        "Review opportunity",
        "General review"
    )
)

# --------------------------------------------------
# 10. Keep useful columns only
# --------------------------------------------------

queue_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action",
    "reason_code",
    "model_score",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "content_age_days",
    "label"
]

action_queue = action_queue[
    [c for c in queue_columns if c in action_queue.columns]
]

print("\nRanked action queue created.")
print("Rows:", len(action_queue))

print("\nTop 10 review queue:")
display(action_queue.head(10))

HF token loaded.
DuckDB connection ready.
Rows available for modeling: 58279
Train rows: 47834
Test rows: 10445
Client overlap: 0

Ranked action queue created.
Rows: 10445

Top 10 review queue:


,rank,client_hash_id,content_hash_id,action,reason_code,model_score,gsc_impressions,gsc_clicks,gsc_avg_position,content_age_days,label
0,1,client_fef1a8f436438636,content_d2f5ff4215ddc515,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,23208.0,165.0,4.085186,232,1
1,2,client_fef1a8f436438636,content_6b4ba5a247ea6100,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,57917.0,424.0,3.842067,232,1
2,3,client_fef1a8f436438636,content_ae68e15ceab3a802,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,38744.0,204.0,10.713736,232,1
3,4,client_fef1a8f436438636,content_4fdb9cd60244859a,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,31806.0,205.0,5.143589,232,1
4,5,client_fef1a8f436438636,content_7960401fa9cfe53b,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,18453.0,196.0,11.947326,232,1
5,6,client_fef1a8f436438636,content_69560b448e635cdc,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,48425.0,185.0,5.435168,232,1
6,7,client_fef1a8f436438636,content_510b0e6411887145,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,24577.0,155.0,12.971640,232,1
7,8,client_fef1a8f436438636,content_a2d8d8c609ff7b0e,Review opportunity,high_model_priority;baseline_visible,1.0,18088.0,143.0,5.560980,162,1
8,9,client_fef1a8f436438636,content_7a02e4bb9d4a379a,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,23187.0,148.0,3.556260,232,1
9,10,client_fef1a8f436438636,content_2cb69ba238f9c395,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,12499.0,133.0,5.996960,232,1


## 2. Intended use and limits


This playbook is intended for content or SEO reviewers who need to decide which pages to inspect first.

The ranked queue is decision-support evidence. It helps prioritize human review; it does not automatically decide that a page needs a refresh or that a page is failing.

The current evidence comes from the February 2026 feature window and March 2026 label window used in the model. The model was evaluated on a held-out client-grouped test split.

The main limits are that the model uses a proxy label, the evaluation is based on one held-out split, and the observed relationships do not establish causation. The queue should therefore be reviewed by a person before any content change is made.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Intended use: prioritize pages for human review.")
print("Automatic publishing or automatic content changes: NOT recommended.")

print("\nModeling rows:", len(action_queue))
print("Unique client-content pairs:",
      action_queue[["client_hash_id", "content_hash_id"]].drop_duplicates().shape[0])

print("\nDecision-support checks:")
print("- Human review required: YES")
print("- Causal conclusion supported: NO")
print("- Automatic refresh decision: NO")
print("- Evaluation basis: held-out client-grouped test split")

Intended use: prioritize pages for human review.
Automatic publishing or automatic content changes: NOT recommended.

Modeling rows: 10445
Unique client-content pairs: 10445

Decision-support checks:
- Human review required: YES
- Causal conclusion supported: NO
- Automatic refresh decision: NO
- Evaluation basis: held-out client-grouped test split


## 3. Human review + the no-go list

A reviewer should check the page context, recent performance, search intent, content quality, and any known business or editorial constraints before acting on a recommendation.

The model score and reason code explain why a page entered the queue, but they do not explain the full cause of a performance change.

The following should not be automated from this model alone: publishing content changes, deleting pages, changing search strategy, declaring a page successful or failing, or treating the model score as causal evidence.

Human review is required before any action is taken.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: create a human-review checklist for the Top-10 queue

review_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action",
    "reason_code",
    "model_score",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "content_age_days",
    "label"
]

available_review_columns = [
    c for c in review_columns
    if c in action_queue.columns
]

top10_review = action_queue[available_review_columns].head(10).copy()

print("Top-10 human review queue:")
display(top10_review)

print("\nHuman review required before action: YES")

print("\nNo-go automation list:")
no_go = [
    "Automatic content publishing",
    "Automatic page deletion",
    "Automatic SEO strategy changes",
    "Automatic declaration that a page is failing",
    "Treating model score as causal evidence"
]

for item in no_go:
    print("-", item)

Top-10 human review queue:


,rank,client_hash_id,content_hash_id,action,reason_code,model_score,gsc_impressions,gsc_clicks,gsc_avg_position,content_age_days,label
0,1,client_fef1a8f436438636,content_d2f5ff4215ddc515,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,23208.0,165.0,4.085186,232,1
1,2,client_fef1a8f436438636,content_6b4ba5a247ea6100,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,57917.0,424.0,3.842067,232,1
2,3,client_fef1a8f436438636,content_ae68e15ceab3a802,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,38744.0,204.0,10.713736,232,1
3,4,client_fef1a8f436438636,content_4fdb9cd60244859a,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,31806.0,205.0,5.143589,232,1
4,5,client_fef1a8f436438636,content_7960401fa9cfe53b,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,18453.0,196.0,11.947326,232,1
5,6,client_fef1a8f436438636,content_69560b448e635cdc,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,48425.0,185.0,5.435168,232,1
6,7,client_fef1a8f436438636,content_510b0e6411887145,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,24577.0,155.0,12.971640,232,1
7,8,client_fef1a8f436438636,content_a2d8d8c609ff7b0e,Review opportunity,high_model_priority;baseline_visible,1.0,18088.0,143.0,5.560980,162,1
8,9,client_fef1a8f436438636,content_7a02e4bb9d4a379a,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,23187.0,148.0,3.556260,232,1
9,10,client_fef1a8f436438636,content_2cb69ba238f9c395,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,12499.0,133.0,5.996960,232,1



Human review required before action: YES

No-go automation list:
- Automatic content publishing
- Automatic page deletion
- Automatic SEO strategy changes
- Automatic declaration that a page is failing
- Treating model score as causal evidence


## 4. Monitoring / retrain triggers

The recommendations should be reviewed if the data distribution or the relationship between the signals and the March outcome changes.

Useful triggers include a material change in the number of eligible pages, changes in the distribution of impressions or clicks, missing GSC data, a sustained drop in Precision@50, or a change in the business workflow that makes the current reason codes or actions unsuitable.

If these changes persist, the model and baseline should be re-evaluated using a fresh validation window before continuing to rely on the queue.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: simple monitoring checks

print("Monitoring snapshot")

print("\nQueue rows:", len(action_queue))

if "gsc_impressions" in action_queue.columns:
    print(
        "Median impressions:",
        round(action_queue["gsc_impressions"].median(), 2)
    )

if "gsc_clicks" in action_queue.columns:
    print(
        "Median clicks:",
        round(action_queue["gsc_clicks"].median(), 2)
    )

if "content_age_days" in action_queue.columns:
    print(
        "Median content age (days):",
        round(action_queue["content_age_days"].median(), 2)
    )

print("\nSuggested review triggers:")
print("- Precision@50 drops materially on a fresh validation window")
print("- GSC availability or feature coverage changes")
print("- Feature distributions change materially")
print("- Reason codes no longer match reviewer needs")
print("- The content workflow changes")

Monitoring snapshot

Queue rows: 10445
Median impressions: 351.0
Median clicks: 1.0
Median content age (days): 73.0

Suggested review triggers:
- Precision@50 drops materially on a fresh validation window
- GSC availability or feature coverage changes
- Feature distributions change materially
- Reason codes no longer match reviewer needs
- The content workflow changes


## 5. Exports for the paper

The ranked action queue is exported so the recommendations can be reused in the research paper.

The export contains the rank, action, reason code, model score, key February signals, content age, and proxy label where available.

The exported queue is an evidence artifact for the analysis. It should not be treated as a final list of pages that must be changed without human review.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 5: export the ranked action queue

import os

output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

export_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action",
    "reason_code",
    "model_score",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "content_age_days",
    "label"
]

export_columns = [
    c for c in export_columns
    if c in action_queue.columns
]

paper_queue = action_queue[export_columns].copy()

output_path = os.path.join(
    output_dir,
    "content_action_queue.csv"
)

paper_queue.to_csv(
    output_path,
    index=False
)

print("Export created:")
print(output_path)

print("Rows exported:", len(paper_queue))
print("Columns exported:", list(paper_queue.columns))

print("\nFirst 5 exported rows:")
display(paper_queue.head())

Export created:
work/outputs/content_action_queue.csv
Rows exported: 10445
Columns exported: ['rank', 'client_hash_id', 'content_hash_id', 'action', 'reason_code', 'model_score', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'content_age_days', 'label']

First 5 exported rows:


,rank,client_hash_id,content_hash_id,action,reason_code,model_score,gsc_impressions,gsc_clicks,gsc_avg_position,content_age_days,label
0,1,client_fef1a8f436438636,content_d2f5ff4215ddc515,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,23208.0,165.0,4.085186,232,1
1,2,client_fef1a8f436438636,content_6b4ba5a247ea6100,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,57917.0,424.0,3.842067,232,1
2,3,client_fef1a8f436438636,content_ae68e15ceab3a802,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,38744.0,204.0,10.713736,232,1
3,4,client_fef1a8f436438636,content_4fdb9cd60244859a,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,31806.0,205.0,5.143589,232,1
4,5,client_fef1a8f436438636,content_7960401fa9cfe53b,Review for refresh,high_model_priority;baseline_visible;baseline_...,1.0,18453.0,196.0,11.947326,232,1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.